# **SCAPES: simplified tutorial**

Train a SCAPES model with the default settings and generate sounds from it. There are three steps: dataprep, training and generation.

The detailed notebook, `tutorial.ipynb`, has the same steps one by one, with the options explained.

## 1. Setup

Run the first code cell first. On Colab it mounts your Google Drive, clones the SCAPES repository and installs it with the steps of the README. Locally it uses the repository this notebook is in, so install the environment first (README, Installation) and open the notebook from inside the repository.

In [ ]:
# Setup. On Colab: clone SCAPES (with its submodule), mount Drive and install it with the README steps.
# Locally: use the repository this notebook is in (the folder above it that holds SCAPES/pipeline.py).
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # branch cloned on Colab

if IN_COLAB:
    from google.colab import drive

    # clone once: the folder /content/SCAPES is kept if the cell runs again
    if not os.path.isdir("/content/SCAPES"):
        subprocess.run(["git", "clone", "--recurse-submodules", "--branch", BRANCH,
                        "https://github.com/cordutie/SCAPES.git", "/content/SCAPES"], check=True)
    os.chdir("/content/SCAPES")
    drive.mount("/content/drive")

    # the install steps of the README (install:env), without the virtual environment
    python_m = [sys.executable, "-m"]
    subprocess.run(python_m + ["pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run(python_m + ["pip", "install", "-r", "requirements.txt",
                               "--extra-index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run(python_m + ["pip", "install", "--no-deps", "msclap==1.3.4",
                               "magenta-rt @ git+https://github.com/magenta/magenta-realtime@694a545e4ba0b88bf1150137b129582166d3e07f"],
                   check=True)
    subprocess.run(python_m + ["pip", "install", "music2latent==0.1.8"], check=True)
    subprocess.run(python_m + ["pip", "install", "-e", "./external/audio_codec_wrappers"], check=True)
    subprocess.run([sys.executable, "external/audio_codec_wrappers/scripts/fetch_weights.py", "--codec", "spectrostream"],
                   check=True)
    REPO = os.getcwd()
else:
    # walk up from the working directory to the repository root
    REPO = os.getcwd()
    while not os.path.exists(os.path.join(REPO, "SCAPES", "pipeline.py")):
        parent = os.path.dirname(REPO)
        if parent == REPO:
            raise FileNotFoundError("Open this notebook from inside the SCAPES repository.")
        REPO = parent

# the repository must be importable as SCAPES and quickstart
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("SCAPES repository:", REPO)

## 2. Parameters

Edit the placeholders in the next cell, or set the environment variables `SCAPES_<NAME>` (for example `SCAPES_DATASET_PATH`) to run the notebook without editing it.

Your dataset folder has `raw/` (your WAV files) and `config/` (`dataprep.gin`, `training.gin` and `cherry_picking.csv`; copy the templates from `SCAPES/config/`). The three generation keys are `CODEC`, `CONDITIONING_RATE` and `CONDITIONING_MEMORY_SECONDS`. They are the settings of the model. `docs/generation_modes.html` lists the rates each codec offers.

In [ ]:
import os

# Paths. Edit the placeholders, or set the environment variable SCAPES_<NAME> instead.
DATASET_PATH = os.environ.get("SCAPES_DATASET_PATH", "path/to/your/dataset")  # folder with raw/ and config/
MODEL_PATH = os.environ.get("SCAPES_MODEL_PATH", "../my_model")               # where the model is saved
SEMANTIC_FLAG = os.environ.get("SCAPES_SEMANTIC_FLAG", "gentle_rain")         # a flag of cherry_picking.csv

# The three generation keys (they set the codec and the geometry of the model)
CODEC = os.environ.get("SCAPES_CODEC", "music2latent")  # encodec24 | spectrostream | encodec48 | music2latent
CONDITIONING_RATE = float(os.environ.get("SCAPES_CONDITIONING_RATE", 10.0))  # Hz
CONDITIONING_MEMORY_SECONDS = float(os.environ.get("SCAPES_CONDITIONING_MEMORY_SECONDS", 0.5))  # seconds

# Generation
NFE = int(os.environ.get("SCAPES_NFE", 16))               # ODE steps: more is slower, usually better
CFG_SCALE = float(os.environ.get("SCAPES_CFG_SCALE", 1.0))  # classifier-free guidance: 1.0 = off (default); above 1 it applies to every step and every codec, and higher gives more typical sounds
SEED = int(os.environ.get("SCAPES_SEED", 0))

## Timing check (about one minute on a GPU)

How fast this machine makes sound with the model size set in `training.gin` and the three generation keys above. The model has random weights (no training, no checkpoint): it makes 10 seconds of sound ten times and prints the average.

In [ ]:
# Timing check: random weights, ten runs of 10 s of sound with 16 ODE steps (fixed here, whatever NFE is set above).
import time
from pathlib import Path

import torch

from SCAPES.data.config_loader import load_training_config
from SCAPES.inference.FlowInference import FlowInference
from SCAPES.pipeline import units_for_seconds

gin = Path(DATASET_PATH, "config", "training.gin")  # model size from training.gin (the template if there is no dataset yet)
size = load_training_config(gin if gin.exists() else Path(REPO, "SCAPES", "config", "training.gin")).size
device = "cuda" if torch.cuda.is_available() else "cpu"
geometry = {"conditioning_rate": CONDITIONING_RATE, "conditioning_memory_seconds": CONDITIONING_MEMORY_SECONDS}
engine = FlowInference.from_config(CODEC, size=size, geometry=geometry, device=device)
units = units_for_seconds(engine.geom, 10)
random_sound = lambda: [torch.randn(1024) for _ in range(units)]  # one random CLAP-like vector per unit

engine.generate_soundscape(random_sound(), nfe=16)  # warm-up (GPU kernels, codec loading): not timed
wall = seconds = 0.0
for _ in range(10):
    start = time.perf_counter()
    audio = engine.generate_soundscape(random_sound(), nfe=16)
    if device == "cuda":
        torch.cuda.synchronize()
    wall += time.perf_counter() - start
    seconds += audio.shape[-1] / engine.sr
per_second = wall / seconds
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (CPU)"
print(f"{gpu}, {CODEC}, model '{size}': {per_second:.2f} s to make 1 s of sound. Real time: {'yes' if per_second < 1.0 else 'no'}.")

## 3. Dataprep

Encodes every file of `raw/` with the codec and computes the semantic annotations (CLAP) that the model is conditioned on.

In [ ]:
from quickstart.wrappers import data_preparation

data_preparation(DATASET_PATH, codec=CODEC, conditioning_rate=CONDITIONING_RATE,
                 conditioning_memory_seconds=CONDITIONING_MEMORY_SECONDS)

In [ ]:
# Free the memory of the step before, without a session reset
import gc

import torch

gc.collect()
torch.cuda.empty_cache()

## 4. Training

Trains the model and saves it in `MODEL_PATH`. Training also builds the semantics folder from `config/cherry_picking.csv`, which the generation step needs. If the dataset is too short for a validation split, the last checkpoint is used.

In [ ]:
from quickstart.wrappers import training_loop

training_loop(DATASET_PATH, MODEL_PATH, codec=CODEC, conditioning_rate=CONDITIONING_RATE,
              conditioning_memory_seconds=CONDITIONING_MEMORY_SECONDS)

In [ ]:
# Free the memory of the step before, without a session reset
import gc

import torch

gc.collect()
torch.cuda.empty_cache()

## 5. Generation

Generates sound from the semantic vectors of one flag of `cherry_picking.csv` (`SEMANTIC_FLAG`). The output has one unit of audio per stored vector of that flag, so its length follows the segment in the CSV. The file is written to `MODEL_PATH/generated/generation.wav`.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Audio, display

from SCAPES.pipeline import run_generation

# one CLAP vector per row of the flag's file; each row is one unit of audio
prompts = list(torch.load(Path(MODEL_PATH) / "semantics" / f"{SEMANTIC_FLAG}.pt"))
wav = run_generation(model_dir=MODEL_PATH, out_path=Path(MODEL_PATH) / "generated", prompts=prompts,
                     seed=SEED, nfe=NFE, cfg_scale=CFG_SCALE)
display(Audio(str(wav)))